In [2]:
#pip install transformers accelerate evaluate datasets peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [3]:
#!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import transformers
import accelerate
import peft
import torchao
import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import AutoImageProcessor
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer
from torchvision.transforms import (
    CenterCrop,
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    Resize,
    ToTensor,
)

print(f"Transformers version: {transformers.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"PEFT version: {peft.__version__}")

Transformers version: 5.15.0
Accelerate version: 1.14.0
PEFT version: 0.20.0


In [2]:
model_checkpoint = "google/vit-base-patch16-224"

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
dataset = load_dataset("ethz/food101", split="train")

README.md:   0%|          | 0.00/16.4k [00:00<?, ?B/s]

data/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  490MB            

data/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  464MB            

data/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

...

data/validation-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/validation-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  426MB            

data/validation-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

In [5]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(len(train_dataset))
print(len(val_dataset))

60600
15150


In [6]:
labels = train_dataset.features["label"].names

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(len(labels))

101


In [7]:
image_processor = AutoImageProcessor.from_pretrained(
    model_checkpoint
)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

In [8]:
normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
train_transforms = Compose(
    [
        RandomResizedCrop(image_processor.size["height"]),
        RandomHorizontalFlip(),
        ToTensor(),
        normalize,
    ]
)

val_transforms = Compose(
    [
        Resize(image_processor.size["height"]),
        CenterCrop(image_processor.size["height"]),
        ToTensor(),
        normalize,
    ]
)


def preprocess_train(example_batch):
    """Apply train_transforms across a batch."""
    example_batch["pixel_values"] = [train_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch


def preprocess_val(example_batch):
    """Apply val_transforms across a batch."""
    example_batch["pixel_values"] = [val_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

In [9]:
train_dataset.set_transform(preprocess_train)
val_dataset.set_transform(preprocess_val)

In [10]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )

In [11]:
model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([101])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([101, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [12]:
print_trainable_parameters(model)

trainable params: 85876325 || all params: 85876325 || trainable%: 100.00


In [13]:
# Freeze the entire ViT backbone
for param in model.vit.parameters():
    param.requires_grad = False

# Classifier remains trainable
for param in model.classifier.parameters():
    param.requires_grad = True

In [14]:
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Trainable params: {trainable_params:,}")
print(f"All params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

Trainable params: 77,669
All params: 85,876,325
Trainable %: 0.0904%


In [15]:
model_name = model_checkpoint.split("/")[-1]
batch_size = 32

args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-lora-food101",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=batch_size,
    fp16=True,
    num_train_epochs=5,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    label_names=["labels"],
)

In [16]:
metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [17]:
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

In [18]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)
train_results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.941764,0.787693,0.795380
2,0.875915,0.734905,0.807525
3,0.767569,0.714583,0.813531
4,0.758958,0.703741,0.815116


/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch,Training Loss,Validation Loss,Accuracy
1,0.941764,0.787693,0.795380
2,0.875915,0.734905,0.807525
3,0.767569,0.714583,0.813531
4,0.758958,0.703741,0.815116
5,0.696761,0.697892,0.818152


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

In [19]:
trainer.evaluate(val_dataset)

Training Loss,Validation Loss,Epoch,Accuracy
0.696761,0.697892,5,0.818152


{'eval_loss': 0.6978915333747864, 'eval_accuracy': 0.8181518151815181}

In [20]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [21]:
trainer.save_model(
    "/content/drive/MyDrive/vit-food101/models/linear_probe"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
import json
import os

results = {
    "method": "linear_probe",
    "model": model_checkpoint,
    "best_accuracy": trainer.evaluate()["eval_accuracy"],
    "eval_loss": trainer.evaluate()["eval_loss"],
    "train_runtime_sec": train_results.metrics["train_runtime"],
    "trainable_params": 77669,
    "total_params": 85876325,
    "trainable_pct": 0.0904,
    "epochs": 5,
    "learning_rate": 1e-3,
    "batch_size": 32,
    "gradient_accumulation_steps": 4,
    "gpu": "NVIDIA T4",
    "seed": 42,
}

Training Loss,Validation Loss,Epoch,Accuracy
0.696761,0.697892,5,0.818152


Training Loss,Validation Loss,Epoch,Accuracy
0.696761,0.697892,5,0.818152


In [23]:
save_dir = "/content/drive/MyDrive/vit-food101/results"
os.makedirs(save_dir, exist_ok=True)

with open(f"{save_dir}/linear_probe_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

In [24]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)

history.to_csv(
    f"{save_dir}/linear_probe_history.csv",
    index=False
)